In [48]:
import csv
import zipfile
import pandas as pd
import json
import groq
from openai import OpenAI
from dotenv import load_dotenv
import os
from collections import Counter

### LOADING THE API KEY FORM ENVIRONMENT VARIABLE

In [3]:
#client = Groq(api_key=api_key)

load_dotenv()
api_key = os.getenv("GroqAPIKey")

client = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")

### LOADING DATA

In [4]:
#Personal lap path
zip_path = r"D:\Veena\Certificate Program in AI and ML\Dataset.zip"
#office lap path
#zip_path = r"E:\Veena\Certificate-Program-in-AIML-IIT-Patna\MODULE 2 - CLASSICAL ML\Dataset.zip"
with zipfile.ZipFile(zip_path) as z:    
    empData = pd.read_csv(z.open("Dataset/employee_data.csv"))

#the orient key word, organises the records using rows instead of columns.
#To store the entries, it uses a list.
empDataList = empData.to_dict(orient="records")


### DEFINING TOOLS

In [52]:
#This function returns the column's min, max, count, mean.
def getStats(column:str) -> str:
    try:
        values = [float(row[column]) for row in empDataList]
        response = json.dumps({
            "status":"success",
            "column":column,
            "min":min(values),
            "max":max(values),
            "mean":round(sum(values)/len(values) , 2),
            "count":len(values)
        })
        return response
    except Exception as e:
        response = json.dumps({
            "status":"errro",
            "message":f"Exception occured in getStats: {e}"
        })
        return response

#Filter the data according to the value and the column provided as parameters
def filterData(column:str,value:str) -> str:
    try:
        matches = [row for row in empDataList if row[column].lower() == value.lower()]
        return json.dumps({
            "status":"success",
            "filter":f"{column}:{value}",
            "result":matches
            })
    except Exception as e:
        return json.dumps({
            "status":"error",
            "message":f"Exception occured in filterData:{e}"
            })     
    
def getGroupedResult(column:str) -> str:
    try:
        counts = Counter(emp[column] for emp in empDataList) 
        return json.dumps(
            {
                "status":"success",
                "column":column,
                "counts":counts
            }
        )
    except Exception as e:
        return json.dumps(
            {
                "status":"error",
                "message":f"Exception in getStats:{e}"
            }
        )

### TESTING THE TOOLS

In [53]:
statsResp = getStats("salary")
filterResp = filterData(column="city",value="chennai")
groupResult = getGroupedResult(column="city")

print(f"Statistics of salary column from employee data: {statsResp}")
print(f"Employees from chennai: {filterResp}")
print(f"groupResult: {groupResult}")




Statistics of salary column from employee data: {"status": "success", "column": "salary", "min": 32332.0, "max": 247670.0, "mean": 131195.77, "count": 1000}
Employees from chennai: {"status": "success", "filter": "city:chennai", "result": [{"name": "Ananya Malhotra", "department": "Sales", "salary": 153631, "experience_years": 1, "city": "Chennai"}, {"name": "Pooja Sharma", "department": "Admin", "salary": 168875, "experience_years": 10, "city": "Chennai"}, {"name": "Ananya Kulkarni", "department": "Marketing", "salary": 96466, "experience_years": 1, "city": "Chennai"}, {"name": "Siddharth Sharma", "department": "HR", "salary": 84173, "experience_years": 20, "city": "Chennai"}, {"name": "Neha Jain", "department": "Legal", "salary": 60310, "experience_years": 5, "city": "Chennai"}, {"name": "Karan Reddy", "department": "Admin", "salary": 96963, "experience_years": 2, "city": "Chennai"}, {"name": "Isha Joshi", "department": "Admin", "salary": 107602, "experience_years": 19, "city": "Chen

In [7]:
print(f"Statistics of salary column from employee data: {statsResp}")

Statistics of salary column from employee data: {"status": "success", "column": "salary", "min": 32332.0, "max": 247670.0, "mean": 131195.77, "count": 1000}


### Creating List of Tools

In [55]:
reportTools=[{
    "type":"function",
    "function":{
        "name":"getStats",
        "description":"Get the statictics(min, max, mean, count) of numerical columns. Columns available:salary,experience_years",
        "parameters":{
            "type":"object",
            "properties":{
                "column":{
                    "description":"the column for which the statistics has to be provided.",
                    "type":"string"
                    }
            },
            "required":["column"]
            
        }
    }
},
{
    "type":"function",
    "function":{
        "name":"filterData",
        "description":"To filter the data according to the column and the values.Available columns: name,department,city",
        "parameters":{
            "type":"object",
            "properties":{
                "column":{
                    "description":"The column for which the filter has to be applied",
                    "type":"string"
                },
                "value":{
                    "description":"The value on which the filter has to be applied for the column",
                    "type":"string"
                }
            },
            "required":["column","value"]
        }
        
        }
},
{
    "type":"function",
    "function":{
        "name":"getGroupedResult",
        "description":"Get the count of each value in the column. Columns available:city,department",
        "parameters":{
            "type":"object",
            "properties":{
                "column":{
                    "description":"the column for which the gouping on uniques values has to be done.",
                    "type":"string"
                    }
            },
            "required":["column"]
            
        }
    }
}
]

availableReportTools={
    "getStats":getStats,
    "filterData":filterData,
    "getGroupedResult":getGroupedResult
}

### Code for "Employee Data Assistant" Agent

In [73]:
def employeeDataAssistant(maxIterations,userMsg,verbose):
    allToolResults=[]
    message=[
        {
            "role":"system",
            "content":"You are a useful assistant for the employee data insights."
        },
        {
            "role":"user",
            "content":userMsg
        }
    ]
    if(verbose):
        print(f"*"*80)
        print(f"User message: {userMsg}")
        print("*"*80)

    for step in range(maxIterations):
        #STEP-1
        response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=message,
            tools=reportTools            
        )

        choice = response.choices[0]
        if verbose:
            print(f"choice: {choice}")

        if choice.finish_reason == 'stop':
            break

        if choice.message.tool_calls:
            message.append(choice.message)
            
            for toolCall in choice.message.tool_calls:
                funcName = toolCall.function.name
                args = json.loads(toolCall.function.arguments)

                if verbose:
                    print(f"Tool name: {funcName}, Arguments: {args}")

                toolResult = availableReportTools[funcName](**args)

                if verbose:
                    print(f"Toolname: {funcName}, Result: {toolResult}")

                message.append({
                    "role":"tool",
                    "tool_call_id":toolCall.id,
                    "content":toolResult
                })   
                allToolResults.append({
                    "tool":funcName,
                    "arguments":args,
                    "result":toolResult
                })
    # PHASE - 2: Generating structured output:
    employeeReport = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role":"system",
                "content":"Generate a structured report based on the data analysis."
            },
            {
                "role":"user",
                "content":f"User Question:{userMsg}\n\nData Gathered:{json.dumps(allToolResults)}\n\nGenerate a report."
            }

        ],
        response_format={
            "type":"json_schema",
            "json_schema":{
                "name":"AnalysisReport",
                "strict":True,
                "schema":{
                    "type":"object",
                    "properties":{
                        "question":{"type":"string"},
                        "answer":{"type":"string"},
                        "key_numbers":{
                            "type":"array",
                            "items":{
                                "type":"string"
                            }
                        },
                            "Confidence":{
                                "type":"string",
                                "description":"high, medium or low"                                
                            }                        
                    },
                    "required":["question","answer","key_numbers","Confidence"],
                    "additionalProperties":False
                }
            }
        }
    )
    print(f"employeeReport:{employeeReport}")

    report = json.loads(employeeReport.choices[0].message.content)
    return report

In [39]:
report = employeeDataAssistant(maxIterations=5,userMsg="What's the salary situation in engineering?",verbose=True)


print("\n📋 STRUCTURED REPORT:")
print(json.dumps(report, indent=2))


********************************************************************************
User message: What's the salary situation in engineering?
********************************************************************************
choice: Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_e624a063-474e-449c-9d5a-dc8c0f3506d3', function=Function(arguments='{"column":"department","value":"Engineering"}', name='filterData'), type='function')], reasoning="We need salary stats for engineering department. Use filterData to get engineering rows, then getStats on salary. Probably combine: filter then stats. There's no direct combined function; we can filter then get stats on filtered dataset? The functions probably apply globally, not chainable. Might need to filter then compute stats on that filtered subset. Perhaps

From the above result, especially, if we look at how LLM reasons, we will understand ho both the tools are called wvwn though, in question, the user didn't explicitly ask to filter or get the numerics for them. The LLM is able to reason that the user wants to know about the the numbers of "Salary" column in "Engineering" department. So, first it filtered the data using department=Engineering, and then got the numbers on the filtered data.

In [40]:
report = employeeDataAssistant(maxIterations=5,userMsg="What's the average salary of a lawyer?",verbose=True)


print("\n📋 STRUCTURED REPORT:")
print(json.dumps(report, indent=2))


********************************************************************************
User message: What's the average salary of a lawyer?
********************************************************************************
choice: Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_9fa0e84b-0360-443b-ac99-84216de28e95', function=Function(arguments='{"column":"department","value":"lawyer"}', name='filterData'), type='function')], reasoning='We need average salary for role "lawyer". The data likely has department or position. The filter column could be "department"? Might be "department" includes "Law". Not sure. Use filterData on column maybe "department" with value "lawyer"? Or maybe "position". Not listed. Available columns for filter: name, department, city. So assume department holds job titles. Use filt

In [57]:
report = employeeDataAssistant(maxIterations=5,userMsg="Where does maximum no.of Sales executives comes from?",verbose=True)


print("\n📋 STRUCTURED REPORT:")
print(json.dumps(report, indent=2))


********************************************************************************
User message: Where does maximum no.of Sales executives comes from?
********************************************************************************
choice: Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_78ba74af-9b36-468e-89d8-6f77cf1b48c4', function=Function(arguments='{"column":"department","value":"Sales"}', name='filterData'), type='function')], reasoning='User asks: "Where does maximum no.of Sales executives comes from?" Likely they want to know which city has the maximum number of Sales executives. Or which department? They say "Sales executives" likely a job title. The data columns: name, department, city. We need count of each value in column maybe department? But they want maximum number of Sales executiv

In [75]:
report = employeeDataAssistant(maxIterations=5,userMsg="Which department has maximum no.of employees?",verbose=True)


print("\n📋 STRUCTURED REPORT:")
print(json.dumps(report, indent=2))


********************************************************************************
User message: Which department has maximum no.of employees?
********************************************************************************
choice: Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_0e5d882c-c018-4813-9c6c-40d3c34355c4', function=Function(arguments='{"column":"department"}', name='getGroupedResult'), type='function')], reasoning='We need to get count per department and find which has max. Use getGroupedResult for department.'))
Tool name: getGroupedResult, Arguments: {'column': 'department'}
Toolname: getGroupedResult, Result: {"status": "success", "column": "department", "counts": {"Sales": 90, "HR": 110, "Marketing": 88, "Operations": 107, "Legal": 91, "IT": 92, "Support": 114, "Finance": 99, "Admin